<div style="text-align: center"><img src="images/ngo_project_image.jpg" alt="NGO"></div>

GoodThought NGO has been a catalyst for positive change, focusing its efforts on education, healthcare, and sustainable development to make a significant difference in communities worldwide. With this mission, GoodThought has orchestrated an array of assignments aimed at uplifting underprivileged populations and fostering long-term growth.

We explore how data-driven insights can direct and enhance these humanitarian efforts. We'll engage with the GoodThought SQL database, which encapsulates detailed records of assignments, funding, impacts, and donor activities from 2010 to 2023. This comprehensive dataset includes:

- **`Assignments`:** Details about each project, including its name, duration (start and end dates), budget, geographical region, and the impact score.
- **`Donations`:** Records of financial contributions, linked to specific donors and assignments, highlighting how financial support is allocated and utilized.
- **`Donors`:** Information on individuals and organizations that fund GoodThought’s projects, including donor types.

The below ERD diagram offers a visual representation of the relationships between these data tables:
<div style="text-align: center"><img src="images/erd.png" alt="ERD" width="50%" height="50%"></div>

We will execute SQL queries to answer two questions:
- What are the top 5 assignments that received the highest total donation amounts?
- What is the highest-impact assignment in each region, and how many donations has it received?

In [1]:
%%capture
%run prep_data.ipynb

### What are the top 5 assignments that received the highest total donation amounts ?

In [2]:
%%sql
SELECT a.assignment_name, 
	a.region, 
	ROUND(sum(d.amount),2) as rounded_total_donation_amount,
	dn.donor_type
FROM assignments a
INNER JOIN donations d on d.assignment_id = a.assignment_id
INNER JOIN donors dn on dn.donor_id = d.donor_id
GROUP BY a.assignment_name, a.region, dn.donor_type
ORDER BY rounded_total_donation_amount desc
LIMIT 5;

 * sqlite:///ngo.db
Done.


assignment_name,region,rounded_total_donation_amount,donor_type
Assignment_3033,East,3840.66,Individual
Assignment_300,West,3133.98,Organization
Assignment_4114,North,2778.57,Organization
Assignment_1765,West,2626.98,Organization
Assignment_268,East,2488.69,Individual


### What is the highest-impact assignment in each region, and how many donations has it received?

In [3]:
%%sql
WITH impact_scores AS (
	SELECT a.assignment_name, 
		a.region, 
		a.impact_score, 
		ROW_NUMBER() OVER (PARTITION BY a.region ORDER BY a.impact_score DESC) as row_num
	FROM assignments a 
),
donation_count as (
	SELECT a.assignment_name, a.region, COUNT(d.donation_id) as num_total_donations
	FROM assignments a
	INNER JOIN donations d on d.assignment_id = a.assignment_id
	GROUP BY a.assignment_name, a.region
)
SELECT dc.assignment_name, dc.region, s.impact_score, dc.num_total_donations
FROM donation_count dc 
INNER JOIN impact_scores s ON s.region = s.region AND s.assignment_name = dc.assignment_name
WHERE s.row_num = 1
ORDER BY s.region ASC

 * sqlite:///ngo.db
Done.


assignment_name,region,impact_score,num_total_donations
Assignment_316,East,10.0,2
Assignment_2253,North,9.99,1
Assignment_2794,West,9.99,2
